# 08 — Distributed Residue Consistency

**Local constraints, noisy links, projection, and cost-aware stability**

This notebook adds a distributed bridge to `residue-manifold-learning`.

It is inspired by distributed fault-tolerant quantum computing (FTQC), including IonQ's *Walking Cat* architecture, but it does **not** simulate trapped-ion hardware directly.

Instead, it builds a minimal constraint-system analogue:

```text
local modules → noisy links → global consistency → projection → cost-aware stability
```

## Core claim

Local constraint validity can remain high while global consistency degrades through noisy links.

Projection can restore observed consistency, but realistic correction has cost and imperfect success.

## Notebook outputs

```text
figures/distributed_residue_graph_clean.png
figures/distributed_residue_graph_noisy.png
figures/cgcs_noise_sweep.png
figures/global_consistency_heatmap.png
figures/cgcs_projection_sweep.png
figures/cgcs_cost_aware_projection.png
figures/cost_aware_projection_phase_diagram.png

results/distributed_residue_consistency.csv
results/distributed_residue_projection.csv
results/distributed_residue_cost_aware_projection.csv
results/distributed_residue_summary.json

docs/notebook_08_distributed_residue_consistency.md
```

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError:
    !pip -q install networkx
    import networkx as nx

SEED = 9423
random.seed(SEED)
np.random.seed(SEED)

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Ready.")
print(f"seed = {SEED}")

## 1. Local residue manifold

For a mod30 prime-residue baseline, admissible residues are:

```text
{1, 7, 11, 13, 17, 19, 23, 29}
```

This means primes greater than 5 must persist inside these admissible residue lanes after excluding multiples of 2, 3, and 5.

In [ ]:
MODULUS = 30
ADMISSIBLE_RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29], dtype=int)
ADMISSIBLE_SET = set(ADMISSIBLE_RESIDUES.tolist())

def residue(x, modulus=MODULUS):
    return int(x % modulus)

def is_admissible_residue(r):
    return int(r % MODULUS) in ADMISSIBLE_SET

def sample_local_residues(n_samples=128, noise=0.0):
    inadmissible = np.array([r for r in range(MODULUS) if r not in ADMISSIBLE_SET], dtype=int)
    values = []
    for _ in range(n_samples):
        if np.random.rand() < noise:
            values.append(int(np.random.choice(inadmissible)))
        else:
            values.append(int(np.random.choice(ADMISSIBLE_RESIDUES)))
    return np.array(values, dtype=int)

def local_coverage_score(samples):
    present = set(int(x % MODULUS) for x in samples if is_admissible_residue(x))
    return len(present.intersection(ADMISSIBLE_SET)) / len(ADMISSIBLE_RESIDUES)

def local_validity_score(samples):
    return float(np.mean([is_admissible_residue(x) for x in samples]))

demo = sample_local_residues(n_samples=32, noise=0.0)
print("demo samples:", demo[:16])
print("coverage:", local_coverage_score(demo))
print("validity:", local_validity_score(demo))

## 2. Distributed graph

Each graph node is a local residue module. Each edge is a link between modules.

A link is consistent when paired module residues satisfy:

```text
(r_i - r_j) mod 30 ∈ allowed differences
```

In [ ]:
def allowed_difference_set(residues=ADMISSIBLE_RESIDUES, modulus=MODULUS):
    diffs = set()
    for a in residues:
        for b in residues:
            diffs.add(int((a - b) % modulus))
    return diffs

ALLOWED_DIFFS = allowed_difference_set()
print("allowed differences:", sorted(ALLOWED_DIFFS))
print("count:", len(ALLOWED_DIFFS))

def make_module_graph(n_modules=12, k_neighbors=4, rewiring=0.15, seed=SEED):
    if n_modules < 4:
        raise ValueError("n_modules must be >= 4")
    k = min(k_neighbors, n_modules - 1)
    if k % 2 == 1:
        k += 1
    return nx.watts_strogatz_graph(n=n_modules, k=k, p=rewiring, seed=seed)

def assign_node_residue_samples(G, n_samples=128, local_noise=0.0):
    for node in G.nodes:
        samples = sample_local_residues(n_samples=n_samples, noise=local_noise)
        G.nodes[node]["samples"] = samples
        G.nodes[node]["representative"] = int(np.random.choice(samples))
        G.nodes[node]["coverage"] = local_coverage_score(samples)
        G.nodes[node]["validity"] = local_validity_score(samples)
    return G

G_demo = make_module_graph()
G_demo = assign_node_residue_samples(G_demo)
print("nodes:", G_demo.number_of_nodes())
print("edges:", G_demo.number_of_edges())
print("mean local coverage:", np.mean([G_demo.nodes[n]["coverage"] for n in G_demo.nodes]))

## 3. Link noise and CGCS

Bridge definition:

```text
CGCS = local_coverage × local_validity × link_consistency × global_stability
```

In [ ]:
def evaluate_links(G, link_noise=0.0, allowed_diffs=ALLOWED_DIFFS, seed=None):
    rng = np.random.default_rng(seed)
    for u, v in G.edges:
        ru = int(G.nodes[u]["representative"] % MODULUS)
        rv = int(G.nodes[v]["representative"] % MODULUS)
        corrupted = bool(rng.random() < link_noise)
        observed_rv = rv
        if corrupted:
            observed_rv = int(rng.integers(0, MODULUS))
        diff = int((ru - observed_rv) % MODULUS)
        consistent = diff in allowed_diffs
        G.edges[u, v]["corrupted"] = corrupted
        G.edges[u, v]["observed_diff"] = diff
        G.edges[u, v]["consistent"] = bool(consistent)
    return G

def link_consistency_score(G):
    if G.number_of_edges() == 0:
        return 1.0
    return float(np.mean([G.edges[e]["consistent"] for e in G.edges]))

def global_stability_score(G):
    H = nx.Graph()
    H.add_nodes_from(G.nodes)
    H.add_edges_from([e for e in G.edges if G.edges[e]["consistent"]])
    if H.number_of_nodes() == 0:
        return 0.0
    largest = max((len(c) for c in nx.connected_components(H)), default=0)
    return float(largest / H.number_of_nodes())

def cgcs_score(G):
    local_coverage = float(np.mean([G.nodes[n]["coverage"] for n in G.nodes]))
    local_validity = float(np.mean([G.nodes[n]["validity"] for n in G.nodes]))
    link_consistency = link_consistency_score(G)
    global_stability = global_stability_score(G)
    cgcs = local_coverage * local_validity * link_consistency * global_stability
    return {
        "local_coverage": local_coverage,
        "local_validity": local_validity,
        "link_consistency": link_consistency,
        "global_stability": global_stability,
        "cgcs": float(cgcs),
    }

G_demo = evaluate_links(G_demo, link_noise=0.0, seed=SEED)
cgcs_score(G_demo)

## 4. Visualize clean vs noisy distributed systems

Blue node intensity represents local coverage. Solid edges are consistent. Dashed edges are inconsistent.

In [ ]:
def draw_distributed_graph(G, title, out_path, seed=SEED):
    pos = nx.spring_layout(G, seed=seed, k=0.85)
    consistent_edges = [e for e in G.edges if G.edges[e].get("consistent", True)]
    inconsistent_edges = [e for e in G.edges if not G.edges[e].get("consistent", True)]
    node_scores = [G.nodes[n].get("coverage", 1.0) for n in G.nodes]

    plt.figure(figsize=(9, 6))
    nx.draw_networkx_nodes(
        G, pos, node_size=720, node_color=node_scores, cmap="Blues",
        vmin=0, vmax=1, linewidths=1.2, edgecolors="black"
    )
    nx.draw_networkx_edges(G, pos, edgelist=consistent_edges, width=2.0, alpha=0.8)
    nx.draw_networkx_edges(G, pos, edgelist=inconsistent_edges, width=2.0, alpha=0.8, style="dashed")
    nx.draw_networkx_labels(G, pos, labels={n: f"M{n}" for n in G.nodes}, font_size=9)

    scores = cgcs_score(G)
    subtitle = (
        f"local={scores['local_coverage']:.2f} | "
        f"links={scores['link_consistency']:.2f} | "
        f"global={scores['global_stability']:.2f} | "
        f"CGCS={scores['cgcs']:.2f}"
    )
    plt.title(title + "\n" + subtitle)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()

G_clean = assign_node_residue_samples(make_module_graph(seed=SEED), n_samples=128, local_noise=0.0)
G_clean = evaluate_links(G_clean, link_noise=0.0, seed=SEED)

G_noisy = assign_node_residue_samples(make_module_graph(seed=SEED), n_samples=128, local_noise=0.0)
G_noisy = evaluate_links(G_noisy, link_noise=0.45, seed=SEED + 1)

draw_distributed_graph(G_clean, "Distributed residue graph: clean links", FIG_DIR / "distributed_residue_graph_clean.png")
draw_distributed_graph(G_noisy, "Distributed residue graph: noisy links", FIG_DIR / "distributed_residue_graph_noisy.png")

## 5. Noise sweep

Sweep link noise and local noise.

In [ ]:
def run_single_experiment(
    n_modules=12, k_neighbors=4, rewiring=0.15, n_samples=128,
    local_noise=0.0, link_noise=0.0, seed=SEED
):
    G = make_module_graph(n_modules=n_modules, k_neighbors=k_neighbors, rewiring=rewiring, seed=seed)
    G = assign_node_residue_samples(G, n_samples=n_samples, local_noise=local_noise)
    G = evaluate_links(G, link_noise=link_noise, seed=seed + 99)
    return cgcs_score(G)

def run_noise_sweep(
    link_noise_values=np.linspace(0, 0.7, 15),
    local_noise_values=(0.0, 0.05, 0.10),
    repeats=25,
    n_modules=12,
):
    rows = []
    for local_noise in local_noise_values:
        for link_noise in link_noise_values:
            for rep in range(repeats):
                scores = run_single_experiment(
                    n_modules=n_modules,
                    local_noise=float(local_noise),
                    link_noise=float(link_noise),
                    seed=SEED + rep * 1000 + int(link_noise * 1000) + int(local_noise * 10000),
                )
                rows.append({"local_noise": float(local_noise), "link_noise": float(link_noise), "repeat": rep, **scores})
    return pd.DataFrame(rows)

df = run_noise_sweep()
csv_path = RESULTS_DIR / "distributed_residue_consistency.csv"
df.to_csv(csv_path, index=False)

summary = (
    df.groupby(["local_noise", "link_noise"], as_index=False)
    .agg({
        "local_coverage": "mean",
        "local_validity": "mean",
        "link_consistency": "mean",
        "global_stability": "mean",
        "cgcs": "mean",
    })
)

print(df.head())
print(f"saved: {csv_path}")
summary.head()

In [ ]:
sub = summary[summary["local_noise"] == 0.0].copy()

plt.figure(figsize=(9, 5.5))
plt.plot(sub["link_noise"], sub["local_coverage"], marker="o", label="local coverage")
plt.plot(sub["link_noise"], sub["local_validity"], marker="o", label="local validity")
plt.plot(sub["link_noise"], sub["link_consistency"], marker="o", label="link consistency")
plt.plot(sub["link_noise"], sub["global_stability"], marker="o", label="global stability")
plt.plot(sub["link_noise"], sub["cgcs"], marker="o", linewidth=3, label="CGCS")
plt.xlabel("link noise")
plt.ylabel("score")
plt.ylim(-0.02, 1.02)
plt.title("Distributed residue consistency under link noise (local_noise=0.0)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
noise_fig = FIG_DIR / "cgcs_noise_sweep.png"
plt.savefig(noise_fig, dpi=180, bbox_inches="tight")
plt.show()
print(f"saved: {noise_fig}")

In [ ]:
pivot = summary.pivot(index="local_noise", columns="link_noise", values="cgcs")

plt.figure(figsize=(9, 4.8))
plt.imshow(pivot.values, aspect="auto", origin="lower")
plt.colorbar(label="CGCS")
plt.xticks(range(len(pivot.columns)), [f"{x:.2f}" for x in pivot.columns], rotation=45)
plt.yticks(range(len(pivot.index)), [f"{x:.2f}" for x in pivot.index])
plt.xlabel("link noise")
plt.ylabel("local noise")
plt.title("Global consistency heatmap")
plt.tight_layout()
heatmap_path = FIG_DIR / "global_consistency_heatmap.png"
plt.savefig(heatmap_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"saved: {heatmap_path}")

## 6. Ideal projection: decoder analogue

Idealized projection:

```text
observe noisy link → detect invalid relation → project to nearest valid residue difference
```

In [ ]:
def nearest_allowed_difference(diff, allowed_diffs=ALLOWED_DIFFS, modulus=MODULUS):
    allowed = np.array(sorted(list(allowed_diffs)), dtype=int)

    def cyclic_distance(a, b):
        raw = abs(int(a) - int(b)) % modulus
        return min(raw, modulus - raw)

    distances = np.array([cyclic_distance(diff, a) for a in allowed])
    return int(allowed[np.argmin(distances)])

def project_inconsistent_links_ideal(G, allowed_diffs=ALLOWED_DIFFS):
    H = G.copy()
    n_projected = 0
    total_edges = H.number_of_edges()

    for u, v in H.edges:
        observed = int(H.edges[u, v].get("observed_diff", 0))
        was_consistent = bool(H.edges[u, v].get("consistent", True))

        if was_consistent:
            H.edges[u, v]["projected"] = False
            H.edges[u, v]["projected_diff"] = observed
            H.edges[u, v]["consistent_after_projection"] = True
        else:
            projected = nearest_allowed_difference(observed, allowed_diffs=allowed_diffs)
            H.edges[u, v]["projected"] = True
            H.edges[u, v]["projected_diff"] = projected
            H.edges[u, v]["consistent_after_projection"] = True
            n_projected += 1

        H.edges[u, v]["consistent"] = bool(H.edges[u, v]["consistent_after_projection"])

    H.graph["projection_rate"] = n_projected / total_edges if total_edges else 0.0
    H.graph["n_projected"] = n_projected
    return H

G_noisy_projected = project_inconsistent_links_ideal(G_noisy)
print("Before projection:", cgcs_score(G_noisy))
print("After projection:", cgcs_score(G_noisy_projected))
print("projection_rate:", G_noisy_projected.graph["projection_rate"])

In [ ]:
def run_projection_experiment(local_noise=0.0, link_noise=0.0, seed=SEED):
    G = make_module_graph(seed=seed)
    G = assign_node_residue_samples(G, n_samples=128, local_noise=local_noise)
    G = evaluate_links(G, link_noise=link_noise, seed=seed + 99)
    before = cgcs_score(G)
    projected = project_inconsistent_links_ideal(G)
    after = cgcs_score(projected)
    return {
        "local_noise": float(local_noise),
        "link_noise": float(link_noise),
        "before_cgcs": before["cgcs"],
        "before_link_consistency": before["link_consistency"],
        "before_global_stability": before["global_stability"],
        "after_cgcs": after["cgcs"],
        "after_link_consistency": after["link_consistency"],
        "after_global_stability": after["global_stability"],
        "projection_gain": after["cgcs"] - before["cgcs"],
        "projection_rate": projected.graph.get("projection_rate", 0.0),
    }

rows = []
for link_noise in np.linspace(0, 0.7, 15):
    for rep in range(25):
        rows.append(run_projection_experiment(link_noise=float(link_noise), seed=SEED + rep * 1000 + int(link_noise * 1000)))

projection_df = pd.DataFrame(rows)
projection_csv = RESULTS_DIR / "distributed_residue_projection.csv"
projection_df.to_csv(projection_csv, index=False)
projection_summary = projection_df.groupby("link_noise", as_index=False).mean(numeric_only=True)

print(projection_summary.head())
print(f"saved: {projection_csv}")

In [ ]:
plt.figure(figsize=(9, 5.5))
plt.plot(projection_summary["link_noise"], projection_summary["before_cgcs"], marker="o", label="CGCS before projection")
plt.plot(projection_summary["link_noise"], projection_summary["after_cgcs"], marker="o", linewidth=3, label="CGCS after ideal projection")
plt.plot(projection_summary["link_noise"], projection_summary["projection_rate"], marker="o", label="projection rate")
plt.xlabel("link noise")
plt.ylabel("score")
plt.ylim(-0.02, 1.05)
plt.title("Ideal projection restores constraint consistency")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
projection_fig = FIG_DIR / "cgcs_projection_sweep.png"
plt.savefig(projection_fig, dpi=180, bbox_inches="tight")
plt.show()
print(f"saved: {projection_fig}")

## 7. Cost-aware and imperfect projection

The previous projection step is an ideal decoder analogue. This section adds:

- `p_success`: probability projection succeeds,
- `alpha`: cost penalty per projected link.

```text
effective_CGCS = CGCS_after × (1 - alpha × projection_rate)
```

In [ ]:
def project_inconsistent_links_imperfect(G, allowed_diffs=ALLOWED_DIFFS, p_success=0.9, seed=None):
    rng = np.random.default_rng(seed)
    H = G.copy()
    n_attempted = 0
    n_succeeded = 0
    total_edges = H.number_of_edges()

    for u, v in H.edges:
        observed = int(H.edges[u, v].get("observed_diff", 0))
        was_consistent = bool(H.edges[u, v].get("consistent", True))

        if was_consistent:
            H.edges[u, v]["projection_attempted"] = False
            H.edges[u, v]["projection_succeeded"] = False
            H.edges[u, v]["projected_diff"] = observed
            H.edges[u, v]["consistent_after_projection"] = True
        else:
            n_attempted += 1
            succeeded = bool(rng.random() < p_success)
            H.edges[u, v]["projection_attempted"] = True
            H.edges[u, v]["projection_succeeded"] = succeeded

            if succeeded:
                projected = nearest_allowed_difference(observed, allowed_diffs=allowed_diffs)
                H.edges[u, v]["projected_diff"] = projected
                H.edges[u, v]["consistent_after_projection"] = True
                n_succeeded += 1
            else:
                H.edges[u, v]["projected_diff"] = observed
                H.edges[u, v]["consistent_after_projection"] = False

        H.edges[u, v]["consistent"] = bool(H.edges[u, v]["consistent_after_projection"])

    H.graph["projection_attempt_rate"] = n_attempted / total_edges if total_edges else 0.0
    H.graph["projection_success_rate"] = n_succeeded / n_attempted if n_attempted else 1.0
    H.graph["n_projection_attempted"] = n_attempted
    H.graph["n_projection_succeeded"] = n_succeeded
    return H

def effective_cgcs(after_cgcs, projection_rate, alpha=0.5):
    penalty = max(0.0, 1.0 - alpha * projection_rate)
    return float(after_cgcs * penalty)

G_test = assign_node_residue_samples(make_module_graph(seed=SEED), n_samples=128, local_noise=0.0)
G_test = evaluate_links(G_test, link_noise=0.5, seed=SEED + 123)
G_test_projected = project_inconsistent_links_imperfect(G_test, p_success=0.9, seed=SEED + 456)

before = cgcs_score(G_test)
after = cgcs_score(G_test_projected)
eff = effective_cgcs(after["cgcs"], G_test_projected.graph["projection_attempt_rate"], alpha=0.5)

print("before:", before)
print("after:", after)
print("projection_attempt_rate:", G_test_projected.graph["projection_attempt_rate"])
print("projection_success_rate:", G_test_projected.graph["projection_success_rate"])
print("effective_cgcs:", eff)

In [ ]:
def run_cost_aware_projection_experiment(
    local_noise=0.0, link_noise=0.0, p_success=0.9, alpha=0.5, seed=SEED
):
    G = make_module_graph(seed=seed)
    G = assign_node_residue_samples(G, n_samples=128, local_noise=local_noise)
    G = evaluate_links(G, link_noise=link_noise, seed=seed + 99)
    before = cgcs_score(G)

    projected = project_inconsistent_links_imperfect(G, p_success=p_success, seed=seed + 199)
    after = cgcs_score(projected)

    projection_rate = projected.graph.get("projection_attempt_rate", 0.0)
    projection_success_rate = projected.graph.get("projection_success_rate", 1.0)
    eff = effective_cgcs(after["cgcs"], projection_rate, alpha=alpha)

    return {
        "local_noise": float(local_noise),
        "link_noise": float(link_noise),
        "p_success": float(p_success),
        "alpha": float(alpha),
        "before_cgcs": before["cgcs"],
        "before_link_consistency": before["link_consistency"],
        "before_global_stability": before["global_stability"],
        "after_cgcs": after["cgcs"],
        "after_link_consistency": after["link_consistency"],
        "after_global_stability": after["global_stability"],
        "projection_rate": float(projection_rate),
        "projection_success_rate": float(projection_success_rate),
        "effective_cgcs": float(eff),
        "projection_gain": float(after["cgcs"] - before["cgcs"]),
        "effective_gain": float(eff - before["cgcs"]),
    }

rows = []
for p_success in (1.0, 0.95, 0.90, 0.80):
    for alpha in (0.25, 0.50, 0.75):
        for link_noise in np.linspace(0, 0.7, 15):
            for rep in range(25):
                rows.append(
                    run_cost_aware_projection_experiment(
                        local_noise=0.0,
                        link_noise=float(link_noise),
                        p_success=float(p_success),
                        alpha=float(alpha),
                        seed=SEED + rep * 1000 + int(link_noise * 1000) + int(p_success * 100) + int(alpha * 100),
                    )
                )

cost_df = pd.DataFrame(rows)
cost_csv = RESULTS_DIR / "distributed_residue_cost_aware_projection.csv"
cost_df.to_csv(cost_csv, index=False)

cost_summary = cost_df.groupby(["link_noise", "p_success", "alpha"], as_index=False).mean(numeric_only=True)

print(cost_summary.head())
print(f"saved: {cost_csv}")

In [ ]:
plot_slice = cost_summary[
    (cost_summary["p_success"] == 0.90)
    & (cost_summary["alpha"] == 0.50)
].copy()

plt.figure(figsize=(9, 5.5))
plt.plot(plot_slice["link_noise"], plot_slice["before_cgcs"], marker="o", label="CGCS before projection")
plt.plot(plot_slice["link_noise"], plot_slice["after_cgcs"], marker="o", label="CGCS after imperfect projection")
plt.plot(plot_slice["link_noise"], plot_slice["effective_cgcs"], marker="o", linewidth=3, label="effective CGCS (cost-aware)")
plt.plot(plot_slice["link_noise"], plot_slice["projection_rate"], marker="o", label="projection rate")
plt.xlabel("link noise")
plt.ylabel("score")
plt.ylim(-0.02, 1.05)
plt.title("Cost-aware projection: stability requires increasing effort")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
cost_fig = FIG_DIR / "cgcs_cost_aware_projection.png"
plt.savefig(cost_fig, dpi=180, bbox_inches="tight")
plt.show()
print(f"saved: {cost_fig}")

In [ ]:
alpha_selected = 0.50
phase = cost_summary[cost_summary["alpha"] == alpha_selected].copy()

pivot = phase.pivot_table(index="p_success", columns="link_noise", values="effective_cgcs", aggfunc="mean")

plt.figure(figsize=(9, 4.8))
plt.imshow(pivot.values, aspect="auto", origin="lower")
plt.colorbar(label="effective CGCS")
plt.xticks(range(len(pivot.columns)), [f"{x:.2f}" for x in pivot.columns], rotation=45)
plt.yticks(range(len(pivot.index)), [f"{x:.2f}" for x in pivot.index])
plt.xlabel("link noise")
plt.ylabel("projection success probability")
plt.title(f"Cost-aware projection phase diagram (alpha={alpha_selected})")
plt.tight_layout()
phase_fig = FIG_DIR / "cost_aware_projection_phase_diagram.png"
plt.savefig(phase_fig, dpi=180, bbox_inches="tight")
plt.show()
print(f"saved: {phase_fig}")

## 8. Summary exports

In [ ]:
baseline = summary[(summary["local_noise"] == 0.0) & (summary["link_noise"] == 0.0)].iloc[0].to_dict()
high_link_noise = summary[(summary["local_noise"] == 0.0) & (summary["link_noise"] == summary["link_noise"].max())].iloc[0].to_dict()
cost_slice_final = plot_slice[plot_slice["link_noise"] == plot_slice["link_noise"].max()].iloc[0].to_dict()

summary_payload = {
    "notebook": "08_distributed_residue_consistency.ipynb",
    "seed": SEED,
    "modulus": MODULUS,
    "admissible_residues": ADMISSIBLE_RESIDUES.tolist(),
    "allowed_differences": sorted(list(ALLOWED_DIFFS)),
    "core_claim": "Local residue validity can remain high while global consistency drops through noisy links. Projection can restore observed consistency, but correction has cost and imperfect success.",
    "cgcs_definition": "CGCS = local_coverage × local_validity × link_consistency × global_stability",
    "effective_cgcs_definition": "effective_CGCS = CGCS_after × (1 - alpha × projection_rate)",
    "baseline": {k: float(v) if isinstance(v, (int, float, np.floating)) else v for k, v in baseline.items()},
    "high_link_noise": {k: float(v) if isinstance(v, (int, float, np.floating)) else v for k, v in high_link_noise.items()},
    "cost_aware_final_slice": {k: float(v) if isinstance(v, (int, float, np.floating)) else v for k, v in cost_slice_final.items()},
    "figures": [
        "figures/distributed_residue_graph_clean.png",
        "figures/distributed_residue_graph_noisy.png",
        "figures/cgcs_noise_sweep.png",
        "figures/global_consistency_heatmap.png",
        "figures/cgcs_projection_sweep.png",
        "figures/cgcs_cost_aware_projection.png",
        "figures/cost_aware_projection_phase_diagram.png",
    ],
    "results": [
        "results/distributed_residue_consistency.csv",
        "results/distributed_residue_projection.csv",
        "results/distributed_residue_cost_aware_projection.csv",
    ],
}

summary_path = RESULTS_DIR / "distributed_residue_summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

print(json.dumps(summary_payload, indent=2)[:1600] + "...")
print(f"saved: {summary_path}")

In [ ]:
md_text = '''# Notebook 08 — Distributed Residue Consistency

**Core claim:** local residue validity can remain high while global consistency drops through noisy links.

## Bridge

Walking Cat-style distributed FTQC suggests a useful constraint-system question:

> when local modules are stable, how much does global coherence depend on links?

RML analogue:

```text
mod30 local residues
→ residue consistency across links
→ graph-level structure persistence
```

## CGCS bridge definition

```text
CGCS = local_coverage × local_validity × link_consistency × global_stability
```

## Projection bridge

Ideal projection:

```text
noisy observed relation → nearest valid residue relation → restored consistency
```

Cost-aware projection:

```text
effective_CGCS = CGCS_after × (1 - alpha × projection_rate)
```

## Takeaway

Local constraint validity is necessary but not sufficient for distributed consistency.

In distributed systems, links become part of the constraint manifold.

Projection adds a decoder-like analogue: noisy observed relations are mapped back toward valid residue constraints.

Cost-aware projection adds the systems-design tradeoff: stability can be maintained, but correction effort increases with link noise.
'''

md_path = DOCS_DIR / "notebook_08_distributed_residue_consistency.md"
md_path.write_text(md_text, encoding="utf-8")
print(md_text)
print(f"saved: {md_path}")

## 9. Optional zip/export block for Colab

In [ ]:
import zipfile

zip_path = Path("notebook_08_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))

## Final interpretation

```text
single module: local validity
many modules: link consistency
active correction: projection
full system: cost-aware global stability
```

Core bridge:

```text
Walking Cat = dynamic constraint manifold
RML = static residue constraint manifold
distributed scaling = consistency across links
decoder analogue = projection back toward valid structure
systems tradeoff = stability requires increasing correction effort
```